# Hangpost Matching Engine — Exploratory Data Analysis

This notebook explores the seed dataset (`data/test_profiles.csv`, N=1000) so we can answer:

1. What does the user population look like? (age, hometown, mutual-friend density)
2. How rich are the structured fields? (interests, liked topics)
3. How sparse is the synthetic relevance label? (how often does `synthesize_relevance` fire?)
4. Do the scoring components correlate? (signals that always agree are redundant)

**Outputs are intentionally cleared** in version control to keep diffs small. Run all cells locally to populate plots:

```bash
pip install -e ".[ml]" pandas matplotlib seaborn jupyter
jupyter notebook notebooks/01_eda.ipynb
```

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 30)

In [ ]:
df = pd.read_csv(REPO_ROOT / "data" / "test_profiles.csv")
print(f"Loaded {len(df):,} profiles, {df.shape[1]} columns")
df.head()

In [ ]:
df.describe(include="all").transpose()

## Age distribution

Age is a first-class scoring signal (10%-per-year step-down ladder, zero at gaps ≥10 years). The shape of this distribution determines how much of the population is mutually "in lane" for any given source.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["age"], bins=range(df["age"].min(), df["age"].max() + 2), ax=ax, kde=True)
ax.set_title("Age distribution (N=1000)")
ax.set_xlabel("age (years)")
plt.tight_layout()
plt.show()

## Hometown distribution

Hometown is a soft matching signal (`location_match`). A long tail means most users will rarely see a hometown match — so this signal is high-precision but low-coverage.

In [ ]:
top_hometowns = df["hometown"].value_counts().head(20)
fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x=top_hometowns.values, y=top_hometowns.index, ax=ax, orient="h")
ax.set_title("Top 20 hometowns")
ax.set_xlabel("profile count")
plt.tight_layout()
plt.show()

print(f"Unique hometowns: {df['hometown'].nunique()}")
print(f"Top 20 cover: {top_hometowns.sum() / len(df):.1%} of profiles")

## Mutual-friend density

Profiles with mutual friends jump into the social-boost lane and rank ahead of everyone else. By design this should be a *rare* event — otherwise the lane swallows everything.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["friends_in_common"], bins=range(df["friends_in_common"].max() + 2), ax=ax)
ax.set_title("Mutual-friend count per profile")
ax.set_xlabel("friends_in_common")
plt.tight_layout()
plt.show()

with_any = (df["friends_in_common"] > 0).mean()
print(f"{with_any:.1%} of profiles have at least one mutual friend.")

## Token richness (interests + liked topics)

Jaccard overlap depends on how many tokens each profile carries. Sparse profiles (1–2 tokens) get noisier overlap scores than rich profiles (5+ tokens).

In [ ]:
def _count_tokens(cell: str) -> int:
    return len([t for t in str(cell).split(";") if t.strip()])

df["n_interests"] = df["hobbies_activities_sports_games_skills_certifications"].map(_count_tokens)
df["n_liked"] = df["interests_likes"].map(_count_tokens)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["n_interests"], ax=axes[0], discrete=True)
axes[0].set_title("Interest tokens per profile")
sns.histplot(df["n_liked"], ax=axes[1], discrete=True)
axes[1].set_title("Liked-topic tokens per profile")
plt.tight_layout()
plt.show()

## Synthetic relevance label rate

How often does `synthesize_relevance` fire? Too rarely → metrics are noisy; too often → metrics are uninformative. We sample 50 random sources and compute the share of all other profiles that get labeled relevant.

In [ ]:
from hangpost_matching import build_queries, load_profiles_from_csv

profiles = load_profiles_from_csv(REPO_ROOT / "data" / "test_profiles.csv")
queries = build_queries(profiles, n_sources=50, seed=42)
rates = [len(rel) / len(cands) for _, cands, rel in queries]

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(rates, bins=20, ax=ax)
ax.set_title("synthesize_relevance label rate per query")
ax.set_xlabel("share of candidates labeled relevant")
plt.tight_layout()
plt.show()

print(f"Median label rate: {pd.Series(rates).median():.1%}")

## Component-score correlations

Signals that always co-fire are redundant — the model can't learn from them independently. We score 1000 random source-candidate pairs and look at component correlations.

In [ ]:
import random
from hangpost_matching import compute_match_score

rng = random.Random(0)
rows = []
for _ in range(1000):
    a, b = rng.sample(profiles, 2)
    bd = compute_match_score(a, b)
    rows.append({
        "interest_overlap": bd.interest_overlap,
        "liked_topic_overlap": bd.liked_topic_overlap,
        "mutual_friends": bd.mutual_friends,
        "location_match": bd.location_match,
        "age_compatibility": bd.age_compatibility,
        "semantic_similarity": bd.semantic_similarity,
    })
components = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(components.corr(), annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Pearson correlation between scoring components (no embeddings)")
plt.tight_layout()
plt.show()

components.describe()

## Takeaways

Use the plots above to update `docs/DATA_CARD.md` after any dataset refresh. In particular check:

- Mutual-friend density — keep ≪10% so the social lane stays a *signal*, not a default.
- Hometown long tail — confirms `location_match` is a high-precision, low-coverage signal.
- Synthetic relevance rate — outside ~5–25% the metrics get noisy or uninformative.
- Component correlations — strong (>|0.5|) correlations indicate redundant features the learned ranker may collapse.